# Pandas: Cleaning & Transforming


**What you'll learn:**
1. Detecting missing values (`isna`, `isnull`)
2. Dropping or filling missing values
3. Changing data types with `astype`
4. Renaming columns and the index
5. Adding, removing, and reordering columns
6. The `.apply()`, `.map()`, and `.applymap()` methods
7. String operations with `.str`
8. Categorical data (for memory efficiency)


Most of real data work is cleaning. Surveys say data scientists spend 60-80% of their time on data preparation. This notebook is the most "practical" of the series.


In [1]:
import pandas as pd
import numpy as np

# Create a deliberately messy dataset
messy_csv = '''name,age,salary,department,join_date
Alice,28,75000,Engineering,2020-01-15
Bob,,65000,Sales,2019-06-30
Charlie,35,,Engineering,2018-03-10
Diana,29,82000,HR,2021-11-22
Eve,42,95000,Engineering,
Frank,31,70000,sales,2020-08-05
Grace,,55000,HR,2022-02-14
Henry,38,105000,ENGINEERING,2017-09-01
'''

with open('messy_employees.csv', 'w') as f:
    f.write(messy_csv)

df = pd.read_csv('messy_employees.csv')
print(df)

      name   age    salary   department   join_date
0    Alice  28.0   75000.0  Engineering  2020-01-15
1      Bob   NaN   65000.0        Sales  2019-06-30
2  Charlie  35.0       NaN  Engineering  2018-03-10
3    Diana  29.0   82000.0           HR  2021-11-22
4      Eve  42.0   95000.0  Engineering         NaN
5    Frank  31.0   70000.0        sales  2020-08-05
6    Grace   NaN   55000.0           HR  2022-02-14
7    Henry  38.0  105000.0  ENGINEERING  2017-09-01


## Missing Values — The Real-World Constant

Any real dataset has missing values. pandas represents them as `NaN` (Not a Number).

### Detecting missing values

In [2]:
# Boolean DataFrame: True wherever a value is missing
print(df.isna())

# isna() and isnull() are identical aliases — use whichever you prefer

    name    age  salary  department  join_date
0  False  False   False       False      False
1  False   True   False       False      False
2  False  False    True       False      False
3  False  False   False       False      False
4  False  False   False       False       True
5  False  False   False       False      False
6  False   True   False       False      False
7  False  False   False       False      False


In [3]:
# How many missing values per column?
print("Missing per column:")
print(df.isna().sum())

# Total missing
print(f"\nTotal missing: {df.isna().sum().sum()}")

# What percentage is missing per column?
print("\nMissing percentage:")
print((df.isna().mean() * 100).round(1))

Missing per column:
name          0
age           2
salary        1
department    0
join_date     1
dtype: int64

Total missing: 4

Missing percentage:
name           0.0
age           25.0
salary        12.5
department     0.0
join_date     12.5
dtype: float64


In [4]:
# Detect rows with ANY missing values
print("Rows with any missing:")
print(df[df.isna().any(axis=1)])

# Rows with ALL values missing (rare but useful)
print("\nRows with all missing:")
print(df[df.isna().all(axis=1)])

Rows with any missing:
      name   age   salary   department   join_date
1      Bob   NaN  65000.0        Sales  2019-06-30
2  Charlie  35.0      NaN  Engineering  2018-03-10
4      Eve  42.0  95000.0  Engineering         NaN
6    Grace   NaN  55000.0           HR  2022-02-14

Rows with all missing:
Empty DataFrame
Columns: [name, age, salary, department, join_date]
Index: []


## Handling Missing Values

You have three main options: drop, fill, or interpolate.

### Option A: Drop missing values

In [5]:
# dropna() removes rows with ANY missing values
clean1 = df.dropna()
print("After dropna() — only complete rows:")
print(clean1)
print(f"\n{len(df)} → {len(clean1)} rows")

After dropna() — only complete rows:
    name   age    salary   department   join_date
0  Alice  28.0   75000.0  Engineering  2020-01-15
3  Diana  29.0   82000.0           HR  2021-11-22
5  Frank  31.0   70000.0        sales  2020-08-05
7  Henry  38.0  105000.0  ENGINEERING  2017-09-01

8 → 4 rows


In [6]:
# Drop only rows where SPECIFIC columns are missing
clean2 = df.dropna(subset=['age', 'salary'])
print("Dropping rows missing age OR salary:")
print(clean2)

Dropping rows missing age OR salary:
    name   age    salary   department   join_date
0  Alice  28.0   75000.0  Engineering  2020-01-15
3  Diana  29.0   82000.0           HR  2021-11-22
4    Eve  42.0   95000.0  Engineering         NaN
5  Frank  31.0   70000.0        sales  2020-08-05
7  Henry  38.0  105000.0  ENGINEERING  2017-09-01


In [7]:
# Drop columns instead of rows (rarely needed)
clean3 = df.dropna(axis=1)
print("After dropping columns with any missing:")
print(clean3)

After dropping columns with any missing:
      name   department
0    Alice  Engineering
1      Bob        Sales
2  Charlie  Engineering
3    Diana           HR
4      Eve  Engineering
5    Frank        sales
6    Grace           HR
7    Henry  ENGINEERING


### Option B: Fill missing values

In [8]:
# Fill all NaN with a single value
filled1 = df.fillna(0)
print("Filled with 0 (often wrong for strings/dates):")
print(filled1)

Filled with 0 (often wrong for strings/dates):
      name   age    salary   department   join_date
0    Alice  28.0   75000.0  Engineering  2020-01-15
1      Bob   0.0   65000.0        Sales  2019-06-30
2  Charlie  35.0       0.0  Engineering  2018-03-10
3    Diana  29.0   82000.0           HR  2021-11-22
4      Eve  42.0   95000.0  Engineering           0
5    Frank  31.0   70000.0        sales  2020-08-05
6    Grace   0.0   55000.0           HR  2022-02-14
7    Henry  38.0  105000.0  ENGINEERING  2017-09-01


In [9]:
# Fill different columns with different values — best practice
filled2 = df.fillna({
    'age': df['age'].median(),
    'salary': df['salary'].mean(),
    'join_date': '2020-01-01'
})
print("Smart fill (median for age, mean for salary, default date):")
print(filled2)

Smart fill (median for age, mean for salary, default date):
      name   age         salary   department   join_date
0    Alice  28.0   75000.000000  Engineering  2020-01-15
1      Bob  33.0   65000.000000        Sales  2019-06-30
2  Charlie  35.0   78142.857143  Engineering  2018-03-10
3    Diana  29.0   82000.000000           HR  2021-11-22
4      Eve  42.0   95000.000000  Engineering  2020-01-01
5    Frank  31.0   70000.000000        sales  2020-08-05
6    Grace  33.0   55000.000000           HR  2022-02-14
7    Henry  38.0  105000.000000  ENGINEERING  2017-09-01


In [10]:
# Forward fill — use the previous value (great for time series)
ts = pd.Series([10, np.nan, np.nan, 15, np.nan, 20])
print("Original:", ts.tolist())
print("ffill:   ", ts.ffill().tolist())
print("bfill:   ", ts.bfill().tolist())

Original: [10.0, nan, nan, 15.0, nan, 20.0]
ffill:    [10.0, 10.0, 10.0, 15.0, 15.0, 20.0]
bfill:    [10.0, 15.0, 15.0, 15.0, 20.0, 20.0]


**Rule of thumb for handling missing data:**

| Situation | Recommended approach |
|---|---|
| < 5% missing | dropna is usually fine |
| 5-30% missing | fill with mean/median/mode |
| > 30% missing | consider dropping the column entirely |
| Time series gaps | ffill or interpolate |
| Categorical missing | fillna with 'Unknown' or the mode |

There's no single right answer — it depends on WHY values are missing and what you're using the data for.

## Changing Data Types

pandas infers types when loading, but it's often wrong or sub-optimal. Use `.astype()` to fix.

In [11]:
print("Current dtypes:")
print(df.dtypes)

Current dtypes:
name           object
age           float64
salary        float64
department     object
join_date      object
dtype: object


In [12]:
# Note: 'age' is float64 because of the NaN values — once we fill, we can make it int
df_clean = df.copy()
df_clean['age'] = df_clean['age'].fillna(df_clean['age'].median()).astype(int)
df_clean['salary'] = df_clean['salary'].fillna(df_clean['salary'].mean()).astype(int)

print("Cleaned dtypes:")
print(df_clean.dtypes)
print("\nResult:")
print(df_clean)

Cleaned dtypes:
name          object
age            int32
salary         int32
department    object
join_date     object
dtype: object

Result:
      name  age  salary   department   join_date
0    Alice   28   75000  Engineering  2020-01-15
1      Bob   33   65000        Sales  2019-06-30
2  Charlie   35   78142  Engineering  2018-03-10
3    Diana   29   82000           HR  2021-11-22
4      Eve   42   95000  Engineering         NaN
5    Frank   31   70000        sales  2020-08-05
6    Grace   33   55000           HR  2022-02-14
7    Henry   38  105000  ENGINEERING  2017-09-01


### Parsing dates

Date strings are stored as objects by default. Convert them with `pd.to_datetime()`.

In [13]:
df_clean['join_date'] = pd.to_datetime(df_clean['join_date'], errors='coerce')
print("After parsing dates:")
print(df_clean.dtypes)
print(df_clean[['name', 'join_date']])

# Now we can extract date parts
print("\nJoin years:")
print(df_clean['join_date'].dt.year)
print("\nDays since joining:")
days = (pd.Timestamp('2026-06-15') - df_clean['join_date']).dt.days
print(days)

After parsing dates:
name                  object
age                    int32
salary                 int32
department            object
join_date     datetime64[ns]
dtype: object
      name  join_date
0    Alice 2020-01-15
1      Bob 2019-06-30
2  Charlie 2018-03-10
3    Diana 2021-11-22
4      Eve        NaT
5    Frank 2020-08-05
6    Grace 2022-02-14
7    Henry 2017-09-01

Join years:
0    2020.0
1    2019.0
2    2018.0
3    2021.0
4       NaN
5    2020.0
6    2022.0
7    2017.0
Name: join_date, dtype: float64

Days since joining:
0    2343.0
1    2542.0
2    3019.0
3    1666.0
4       NaN
5    2140.0
6    1582.0
7    3209.0
Name: join_date, dtype: float64


## String Operations with `.str`

Real text columns are messy: inconsistent capitalization, extra spaces, mixed formats. The `.str` accessor gives you string methods that work across the whole column at once.

In [14]:
# Notice the 'department' column has inconsistent capitalization
print("Departments:", df_clean['department'].unique())

# Normalize: lowercase, then capitalize
df_clean['department'] = df_clean['department'].str.strip().str.title()
print("Cleaned:    ", df_clean['department'].unique())

Departments: ['Engineering' 'Sales' 'HR' 'sales' 'ENGINEERING']
Cleaned:     ['Engineering' 'Sales' 'Hr']


In [15]:
# More string methods — most Python string methods work via .str
sample = pd.Series(['  Alice  ', 'BOB', 'charlie', 'Diana  ', '  eve'])

print("Original:           ", sample.tolist())
print("Strip whitespace:   ", sample.str.strip().tolist())
print("Lower:              ", sample.str.lower().tolist())
print("Upper:              ", sample.str.upper().tolist())
print("Title:              ", sample.str.title().tolist())
print("Length:             ", sample.str.len().tolist())
print("Contains 'a' (case-insensitive):", sample.str.contains('a', case=False).tolist())

Original:            ['  Alice  ', 'BOB', 'charlie', 'Diana  ', '  eve']
Strip whitespace:    ['Alice', 'BOB', 'charlie', 'Diana', 'eve']
Lower:               ['  alice  ', 'bob', 'charlie', 'diana  ', '  eve']
Upper:               ['  ALICE  ', 'BOB', 'CHARLIE', 'DIANA  ', '  EVE']
Title:               ['  Alice  ', 'Bob', 'Charlie', 'Diana  ', '  Eve']
Length:              [9, 3, 7, 7, 5]
Contains 'a' (case-insensitive): [True, False, True, True, False]


In [16]:
# Splitting strings
emails = pd.Series([
    'alice@gmail.com',
    'bob@yahoo.com',
    'charlie@gmail.com',
    'diana@hotmail.com'
])

# Get the domain (part after @)
domains = emails.str.split('@').str[1]
print("Domains:", domains.tolist())

# Even shorter — extract pattern
domains2 = emails.str.extract(r'@(.+)')
print("Domains (regex):")
print(domains2)

Domains: ['gmail.com', 'yahoo.com', 'gmail.com', 'hotmail.com']
Domains (regex):
             0
0    gmail.com
1    yahoo.com
2    gmail.com
3  hotmail.com


In [17]:
# Replace patterns
phones = pd.Series([
    '+1-555-1234',
    '(555) 555-9999',
    '555.123.4567',
    '5551234567'
])

# Remove everything that isn't a digit
clean_phones = phones.str.replace(r'\D', '', regex=True)
print("Original:")
print(phones)
print("\nDigits only:")
print(clean_phones)

Original:
0       +1-555-1234
1    (555) 555-9999
2      555.123.4567
3        5551234567
dtype: object

Digits only:
0      15551234
1    5555559999
2    5551234567
3    5551234567
dtype: object


## Custom Transformations with `.apply()` and `.map()`

When built-in methods don't fit, write your own logic.

### `.map()` — for Series only

`.map()` applies a function or dict to each element of a Series.

In [18]:
# Using a dict to map values
size_map = {'XS': 0, 'S': 1, 'M': 2, 'L': 3, 'XL': 4}
sizes = pd.Series(['S', 'M', 'L', 'M', 'XL'])

mapped = sizes.map(size_map)
print("Mapped to numbers:")
print(mapped)

Mapped to numbers:
0    1
1    2
2    3
3    2
4    4
dtype: int64


In [19]:
# Using a function
def categorize_age(age):
    if age < 25:
        return 'Young'
    elif age < 40:
        return 'Mid-career'
    else:
        return 'Senior'

df_clean['age_group'] = df_clean['age'].map(categorize_age)
print(df_clean[['name', 'age', 'age_group']])

      name  age   age_group
0    Alice   28  Mid-career
1      Bob   33  Mid-career
2  Charlie   35  Mid-career
3    Diana   29  Mid-career
4      Eve   42      Senior
5    Frank   31  Mid-career
6    Grace   33  Mid-career
7    Henry   38  Mid-career


### `.apply()` — for both Series and DataFrames

`.apply()` is more flexible — works on whole rows or columns of a DataFrame.

In [20]:
# Apply a function to each value in a Series
df_clean['salary_k'] = df_clean['salary'].apply(lambda x: f"${x/1000:.0f}k")
print(df_clean[['name', 'salary', 'salary_k']])

      name  salary salary_k
0    Alice   75000     $75k
1      Bob   65000     $65k
2  Charlie   78142     $78k
3    Diana   82000     $82k
4      Eve   95000     $95k
5    Frank   70000     $70k
6    Grace   55000     $55k
7    Henry  105000    $105k


In [21]:
# Apply a function ROW-WISE (axis=1)
def employee_summary(row):
    return f"{row['name']} ({row['department']}, {row['age']}y)"

df_clean['summary'] = df_clean.apply(employee_summary, axis=1)
print(df_clean[['name', 'summary']].head())

      name                     summary
0    Alice    Alice (Engineering, 28y)
1      Bob            Bob (Sales, 33y)
2  Charlie  Charlie (Engineering, 35y)
3    Diana             Diana (Hr, 29y)
4      Eve      Eve (Engineering, 42y)


In [22]:
# Apply across a whole COLUMN (axis=0, the default)
print("Column statistics with apply:")
print(df_clean[['age', 'salary']].apply(lambda col: col.max() - col.min()))

Column statistics with apply:
age          14
salary    50000
dtype: int64


**When to use which:**

| Method | Works on | Speed | Use when |
|---|---|---|---|
| `.map()` | Series only | Fast for dicts | Replacing values from a lookup |
| `.apply()` | Series or DataFrame | Slower (Python loop) | Custom logic |
| Vectorized ops | Anything | Fastest | Anything you can express with operators |

**`.apply()` is slow** for large datasets. Always check if you can do it with vectorized operations first. `df['col'] * 2` is 100x faster than `df['col'].apply(lambda x: x * 2)`.

## Renaming Columns and Reordering

In [23]:
# Rename specific columns
df_renamed = df_clean.rename(columns={
    'name': 'employee_name',
    'salary': 'annual_salary'
})
print("After rename:")
print(df_renamed.columns.tolist())

After rename:
['employee_name', 'age', 'annual_salary', 'department', 'join_date', 'age_group', 'salary_k', 'summary']


In [24]:
# Rename all columns at once (clean them up)
df_renamed = df_clean.copy()
df_renamed.columns = [col.lower().replace(' ', '_') for col in df_renamed.columns]
print("All lowercased with underscores:")
print(df_renamed.columns.tolist())

All lowercased with underscores:
['name', 'age', 'salary', 'department', 'join_date', 'age_group', 'salary_k', 'summary']


In [25]:
# Reorder columns by selecting in the new order
df_reordered = df_clean[['name', 'department', 'age', 'salary', 'join_date', 'age_group']]
print(df_reordered.head())

      name   department  age  salary  join_date   age_group
0    Alice  Engineering   28   75000 2020-01-15  Mid-career
1      Bob        Sales   33   65000 2019-06-30  Mid-career
2  Charlie  Engineering   35   78142 2018-03-10  Mid-career
3    Diana           Hr   29   82000 2021-11-22  Mid-career
4      Eve  Engineering   42   95000        NaT      Senior


## Adding and Dropping Columns

In [26]:
# Add a new column — just assign to it
df_clean['salary_usd_thousands'] = df_clean['salary'] / 1000
print(df_clean[['name', 'salary', 'salary_usd_thousands']].head())

      name  salary  salary_usd_thousands
0    Alice   75000                75.000
1      Bob   65000                65.000
2  Charlie   78142                78.142
3    Diana   82000                82.000
4      Eve   95000                95.000


In [27]:
# Add based on a condition
df_clean['high_earner'] = df_clean['salary'] > 80000
print(df_clean[['name', 'salary', 'high_earner']])

      name  salary  high_earner
0    Alice   75000        False
1      Bob   65000        False
2  Charlie   78142        False
3    Diana   82000         True
4      Eve   95000         True
5    Frank   70000        False
6    Grace   55000        False
7    Henry  105000         True


In [28]:
# Drop columns
df_smaller = df_clean.drop(columns=['summary', 'salary_k'])
print("Remaining columns:", df_smaller.columns.tolist())

# Drop rows (less common — usually you'd filter)
df_no_eve = df_clean.drop(index=4)
print(f"\nRows: {len(df_clean)} → {len(df_no_eve)}")

Remaining columns: ['name', 'age', 'salary', 'department', 'join_date', 'age_group', 'salary_usd_thousands', 'high_earner']

Rows: 8 → 7


## Categorical Data — Memory and Speed Boost

For columns with a limited set of repeated values (like 'department' with only 3-4 unique options), convert to `category` dtype. This saves memory and speeds up operations.

In [29]:
# Check memory usage before
print("Memory before:")
print(df_clean.memory_usage(deep=True))

# Convert string columns with few unique values to category
df_clean['department'] = df_clean['department'].astype('category')
df_clean['age_group'] = df_clean['age_group'].astype('category')

print("\nMemory after:")
print(df_clean.memory_usage(deep=True))

Memory before:
Index                   132
name                    430
age                      32
salary                   32
department              450
join_date                64
age_group               468
salary_k                425
summary                 552
salary_usd_thousands     64
high_earner               8
dtype: int64

Memory after:
Index                   132
name                    430
age                      32
salary                   32
department              281
join_date                64
age_group               230
salary_k                425
summary                 552
salary_usd_thousands     64
high_earner               8
dtype: int64


For a small DataFrame the savings are minor, but on 1 million rows with a 'country' column of 50 unique values, the memory savings can be 10x or more.

Categories also enable ordered operations:

In [30]:
# Create an ORDERED categorical (useful for things like sizes, grades)
df_clean['age_group'] = pd.Categorical(
    df_clean['age_group'],
    categories=['Young', 'Mid-career', 'Senior'],
    ordered=True
)

# Now you can compare them
print("Anyone older than 'Young'?")
print(df_clean[df_clean['age_group'] > 'Young'][['name', 'age_group']])

Anyone older than 'Young'?
      name   age_group
0    Alice  Mid-career
1      Bob  Mid-career
2  Charlie  Mid-career
3    Diana  Mid-career
4      Eve      Senior
5    Frank  Mid-career
6    Grace  Mid-career
7    Henry  Mid-career


## A Complete Cleaning Workflow

Putting it all together — what a real "clean a CSV" task looks like.

In [32]:
# Start fresh from the original messy CSV
df = pd.read_csv('messy_employees.csv')
print("BEFORE:")
print(df)
print("\nInfo:")
df.info()

BEFORE:
      name   age    salary   department   join_date
0    Alice  28.0   75000.0  Engineering  2020-01-15
1      Bob   NaN   65000.0        Sales  2019-06-30
2  Charlie  35.0       NaN  Engineering  2018-03-10
3    Diana  29.0   82000.0           HR  2021-11-22
4      Eve  42.0   95000.0  Engineering         NaN
5    Frank  31.0   70000.0        sales  2020-08-05
6    Grace   NaN   55000.0           HR  2022-02-14
7    Henry  38.0  105000.0  ENGINEERING  2017-09-01

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   name        8 non-null      object 
 1   age         6 non-null      float64
 2   salary      7 non-null      float64
 3   department  8 non-null      object 
 4   join_date   7 non-null      object 
dtypes: float64(2), object(3)
memory usage: 452.0+ bytes


In [33]:
# Full cleaning pipeline
clean = (
    df
    .assign(
        # Fill missing ages with median, convert to int
        age=lambda d: d['age'].fillna(d['age'].median()).astype(int),
        # Fill missing salaries with mean, convert to int
        salary=lambda d: d['salary'].fillna(d['salary'].mean()).astype(int),
        # Normalize department capitalization
        department=lambda d: d['department'].str.strip().str.title(),
        # Parse join_date (errors='coerce' turns bad values into NaT)
        join_date=lambda d: pd.to_datetime(d['join_date'], errors='coerce'),
    )
    .assign(
        # Add derived columns
        tenure_years=lambda d: (pd.Timestamp('2026-06-15') - d['join_date']).dt.days / 365.25,
        salary_band=lambda d: pd.cut(
            d['salary'],
            bins=[0, 60000, 80000, 100000, float('inf')],
            labels=['Low', 'Mid', 'High', 'Top']
        ),
    )
    .dropna(subset=['join_date'])    # drop rows still missing dates
)

print("AFTER:")
print(clean)
print("\nInfo:")
clean.info()

AFTER:
      name  age  salary   department  join_date  tenure_years salary_band
0    Alice   28   75000  Engineering 2020-01-15      6.414784         Mid
1      Bob   33   65000        Sales 2019-06-30      6.959617         Mid
2  Charlie   35   78142  Engineering 2018-03-10      8.265572         Mid
3    Diana   29   82000           Hr 2021-11-22      4.561259        High
5    Frank   31   70000        Sales 2020-08-05      5.859001         Mid
6    Grace   33   55000           Hr 2022-02-14      4.331280         Low
7    Henry   38  105000  Engineering 2017-09-01      8.785763         Top

Info:
<class 'pandas.core.frame.DataFrame'>
Index: 7 entries, 0 to 7
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   name          7 non-null      object        
 1   age           7 non-null      int32         
 2   salary        7 non-null      int32         
 3   department    7 non-null      object        

**Method chaining** (the `.assign()` pattern above) is a clean way to express a cleaning pipeline. Each step is clear, intermediate variables aren't created, and the whole transformation reads top-to-bottom.

## Practice Exercises

In [35]:
# Setup for exercises — a messier dataset
exercise_csv = '''product,category,price,units_sold,launch_date
Laptop, electronics, 1200, 30, 2023-03-15
phone,Electronics,800,75,2022-06-30
TABLET, electronics,,50,2024-01-10
Monitor,electronics,300,,2023-11-22
Chair,Furniture,150,200,
Desk,furniture,400,80,2022-09-05
Lamp,furniture,45,500,2024-02-14
'''

with open('products.csv', 'w') as f:
    f.write(exercise_csv)

products = pd.read_csv('products.csv')
print(products)

   product      category   price  units_sold  launch_date
0   Laptop   electronics  1200.0        30.0   2023-03-15
1    phone   Electronics   800.0        75.0   2022-06-30
2   TABLET   electronics     NaN        50.0   2024-01-10
3  Monitor   electronics   300.0         NaN   2023-11-22
4    Chair     Furniture   150.0       200.0          NaN
5     Desk     furniture   400.0        80.0   2022-09-05
6     Lamp     furniture    45.0       500.0   2024-02-14


### Exercise 1
Identify which columns have missing values and how many.

### Exercise 2
Clean the data:
- Strip whitespace from `product` and `category` columns
- Make `category` consistent (Title case)
- Fill missing `price` with the median price
- Fill missing `units_sold` with 0
- Drop rows missing `launch_date`
- Convert `launch_date` to datetime

### Exercise 3
Add a new column `revenue` = price × units_sold. Then add a column `price_tier` with values 'Cheap' (< $200), 'Medium' (200-1000), or 'Premium' (> $1000) using `apply` or `pd.cut`.

### Exercise 4
Rename all columns to be UPPERCASE. Then convert `category` to a categorical dtype for memory efficiency.

### Exercise 5 (challenge)
Use the `.str` accessor and regex to extract just the year from `launch_date` as a new column called `launch_year`. (Two ways to do this — try both: `.str.extract()` and `.dt.year` after parsing as datetime.)

In [36]:
# Exercise 1
print("Missing per column:")
print(products.isna().sum())

Missing per column:
product        0
category       0
price          1
units_sold     1
launch_date    1
dtype: int64


In [37]:
# Exercise 2
clean = products.copy()

# Strip + normalize
clean['product'] = clean['product'].str.strip()
clean['category'] = clean['category'].str.strip().str.title()

# Fill missing
clean['price'] = clean['price'].fillna(clean['price'].median())
clean['units_sold'] = clean['units_sold'].fillna(0).astype(int)

# Drop rows missing dates
clean = clean.dropna(subset=['launch_date'])

# Parse dates — strip whitespace first since CSV had leading spaces
clean['launch_date'] = pd.to_datetime(clean['launch_date'].str.strip())

print(clean)
print("\nInfo:")
clean.info()

   product     category   price  units_sold launch_date
0   Laptop  Electronics  1200.0          30  2023-03-15
1    phone  Electronics   800.0          75  2022-06-30
2   TABLET  Electronics   350.0          50  2024-01-10
3  Monitor  Electronics   300.0           0  2023-11-22
5     Desk    Furniture   400.0          80  2022-09-05
6     Lamp    Furniture    45.0         500  2024-02-14

Info:
<class 'pandas.core.frame.DataFrame'>
Index: 6 entries, 0 to 6
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   product      6 non-null      object        
 1   category     6 non-null      object        
 2   price        6 non-null      float64       
 3   units_sold   6 non-null      int32         
 4   launch_date  6 non-null      datetime64[ns]
dtypes: datetime64[ns](1), float64(1), int32(1), object(2)
memory usage: 264.0+ bytes


In [38]:
# Exercise 3
clean['revenue'] = clean['price'] * clean['units_sold']

clean['price_tier'] = pd.cut(
    clean['price'],
    bins=[0, 200, 1000, float('inf')],
    labels=['Cheap', 'Medium', 'Premium']
)
print(clean[['product', 'price', 'revenue', 'price_tier']])

   product   price  revenue price_tier
0   Laptop  1200.0  36000.0    Premium
1    phone   800.0  60000.0     Medium
2   TABLET   350.0  17500.0     Medium
3  Monitor   300.0      0.0     Medium
5     Desk   400.0  32000.0     Medium
6     Lamp    45.0  22500.0      Cheap


In [39]:
# Exercise 4
clean.columns = clean.columns.str.upper()
clean['CATEGORY'] = clean['CATEGORY'].astype('category')
print("Columns:", clean.columns.tolist())
print("\nDtypes:")
print(clean.dtypes)

Columns: ['PRODUCT', 'CATEGORY', 'PRICE', 'UNITS_SOLD', 'LAUNCH_DATE', 'REVENUE', 'PRICE_TIER']

Dtypes:
PRODUCT                object
CATEGORY             category
PRICE                 float64
UNITS_SOLD              int32
LAUNCH_DATE    datetime64[ns]
REVENUE               float64
PRICE_TIER           category
dtype: object


In [40]:
# Exercise 5

# Method 1: using .dt.year (after parsing to datetime — already done)
year_method1 = clean['LAUNCH_DATE'].dt.year
print("Method 1 (.dt.year):")
print(year_method1.tolist())

# Method 2: as string with regex
year_method2 = clean['LAUNCH_DATE'].astype(str).str.extract(r'(\d{4})')
print("\nMethod 2 (regex):")
print(year_method2)

Method 1 (.dt.year):
[2023, 2022, 2024, 2023, 2022, 2024]

Method 2 (regex):
      0
0  2023
1  2022
2  2024
3  2023
5  2022
6  2024
